In [110]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [111]:
fin_txn_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\bronze\finance_transactions.csv")

In [112]:
fin_txn_df

,transaction_id,user_id,date,amount,description,merchant_name,location,payment_mode,category
0,TXN0001,U019,11/03/2024,6614.32,Starbucks *Order#99,Starbucks,BBSR,UPI,other
1,TXN0002,U017,2024/01/31,14567.58,Amazon purchase,Amazon,local shop,Cash,refund
2,TXN0003,U007,2024-01-13,12937.22,Local Shop purchase,Local Shop,NaN,UPI,shopping
3,TXN0004,U013,2024-02-13,5755.79,Starbucks !!!,Starbucks,BBSR,Card,shopping
4,TXN0005,U018,2024-01-13,29978.48,Apple purchase,Apple,BBSR,UPI,refund
...,...,...,...,...,...,...,...,...,...
995,TXN0996,U015,2024/02/22,10566.02,Big Bazaar,Big Bazaar,NaN,Card,refund
996,TXN0997,U008,13/03/2024,7355.87,Local Shop *Order#99,Local Shop,NaN,UPI,electronics
997,TXN0998,U011,2024-01-16,12636.47,DMart 0412,DMart,NaN,UPI,refund
998,TXN0999,U006,16-01-2024,22967.63,Starbucks ???,Starbucks,Bhubaneswar,UPI,Food & Beverage


## 1. date

In [113]:
fin_txn_df['date'] = pd.to_datetime(fin_txn_df['date'], dayfirst=True, format='mixed')

## 2. amount

In [114]:
fin_txn_df['amount'].describe()

count     1000.000000
mean     13600.930230
std      10333.769082
min     -29091.790000
25%       6558.267500
50%      14256.930000
75%      21786.237500
max      29994.640000
Name: amount, dtype: float64

In [115]:
fin_txn_df['amount'] = np.abs(fin_txn_df['amount'])

In [116]:
bins = [31.59, 8500, 18000, 23000, 29994.64]
labels = ['Low', 'Medium-Low', 'Medium-High', 'High']

fin_txn_df['user_category_new'] = pd.cut(fin_txn_df['amount'], bins=bins, labels=labels, include_lowest=True)

fin_txn_df['user_category_new'].value_counts()

user_category_new
Medium-Low     337
Low            282
High           211
Medium-High    170
Name: count, dtype: int64

## 3. Desc

In [117]:
fin_txn_df['description'].unique()

array(['Starbucks *Order#99', 'Amazon  purchase', 'Local Shop  purchase',
       'Starbucks !!!', 'Apple  purchase', 'Big Bazaar  0412', 'Flipkart',
       'Starbucks', 'Unknown Store ???', 'Apple  0412', 'DMart ???',
       'Amazon', 'Swiggy', 'Flipkart *Order#99', 'DMart  0412',
       'Amazon *Order#99', 'Flipkart ???', 'DMart', 'Swiggy ???',
       'Local Shop  0412', 'DMart *Order#99', 'Local Shop ???',
       'Flipkart  0412', 'Starbucks  0412', 'Unknown Store !!!',
       'Big Bazaar  purchase', 'Local Shop', 'Unknown Store  0412',
       'Apple ???', 'Flipkart  fee', 'OLA ???', 'Amazon  0412',
       'Local Shop !!!', 'Unknown Store  fee', 'Apple',
       'Unknown Store *Order#99', 'Unknown Store', 'OLA  0412',
       'Amazon ???', 'Local Shop *Order#99', 'Swiggy *Order#99',
       'Starbucks  fee', 'OLA  fee', 'Apple  fee', 'Big Bazaar *Order#99',
       'Amazon  fee', 'OLA *Order#99', 'Swiggy  purchase', 'Swiggy !!!',
       'OLA', 'Swiggy  0412', 'DMart  purchase', 'Swiggy  

In [118]:
brands = ['Starbucks', 'Amazon', 'Apple', 'Flipkart', 'DMart', 
          'Swiggy', 'OLA', 'Big Bazaar', 'Local Shop', 'Unknown Store']

pattern = '|'.join(brands)
fin_txn_df['clean_brand'] = fin_txn_df['description'].str.extract(f'({pattern})', expand=False)

fin_txn_df['clean_brand'].value_counts()

clean_brand
Apple            115
Amazon           113
Flipkart         108
Big Bazaar       106
Unknown Store    100
OLA               98
DMart             96
Local Shop        90
Swiggy            88
Starbucks         86
Name: count, dtype: int64

In [119]:
fin_txn_df['merchant_name'] = fin_txn_df['clean_brand'].replace('Unknown Store', 'Local Shop')
fin_txn_df['merchant_name'].nunique()

9

In [120]:
fin_txn_df['payment_mode'].value_counts()

payment_mode
Card    361
Cash    338
UPI     301
Name: count, dtype: int64

## 4. Refund

In [121]:
fin_txn_df['category'].nunique()

8

In [122]:
fin_txn_df['category'].value_counts()

category
Food & Beverage    131
shopping           128
food               123
electronics        121
refund             118
grocery            118
transport          112
other              103
Name: count, dtype: int64

In [123]:
fin_txn_df['category'] == 'refund'

0      False
1       True
2      False
3      False
4       True
       ...  
995     True
996    False
997     True
998    False
999    False
Name: category, Length: 1000, dtype: bool

In [124]:
fin_txn_df['is_refund'] = np.where(fin_txn_df['category'] == 'refund', 1, 0)

In [125]:
fin_txn_df

,transaction_id,user_id,date,amount,description,merchant_name,location,payment_mode,category,user_category_new,clean_brand,is_refund
0,TXN0001,U019,2024-03-11,6614.32,Starbucks *Order#99,Starbucks,BBSR,UPI,other,Low,Starbucks,0
1,TXN0002,U017,2024-01-31,14567.58,Amazon purchase,Amazon,local shop,Cash,refund,Medium-Low,Amazon,1
2,TXN0003,U007,2024-01-13,12937.22,Local Shop purchase,Local Shop,NaN,UPI,shopping,Medium-Low,Local Shop,0
3,TXN0004,U013,2024-02-13,5755.79,Starbucks !!!,Starbucks,BBSR,Card,shopping,Low,Starbucks,0
4,TXN0005,U018,2024-01-13,29978.48,Apple purchase,Apple,BBSR,UPI,refund,High,Apple,1
...,...,...,...,...,...,...,...,...,...,...,...,...
995,TXN0996,U015,2024-02-22,10566.02,Big Bazaar,Big Bazaar,NaN,Card,refund,Medium-Low,Big Bazaar,1
996,TXN0997,U008,2024-03-13,7355.87,Local Shop *Order#99,Local Shop,NaN,UPI,electronics,Low,Local Shop,0
997,TXN0998,U011,2024-01-16,12636.47,DMart 0412,DMart,NaN,UPI,refund,Medium-Low,DMart,1
998,TXN0999,U006,2024-01-16,22967.63,Starbucks ???,Starbucks,Bhubaneswar,UPI,Food & Beverage,Medium-High,Starbucks,0


In [126]:
# Already exist in the user db
fin_txn_df = fin_txn_df.drop(columns={'location', 'description'})

In [127]:
fin_txn_df

,transaction_id,user_id,date,amount,merchant_name,payment_mode,category,user_category_new,clean_brand,is_refund
0,TXN0001,U019,2024-03-11,6614.32,Starbucks,UPI,other,Low,Starbucks,0
1,TXN0002,U017,2024-01-31,14567.58,Amazon,Cash,refund,Medium-Low,Amazon,1
2,TXN0003,U007,2024-01-13,12937.22,Local Shop,UPI,shopping,Medium-Low,Local Shop,0
3,TXN0004,U013,2024-02-13,5755.79,Starbucks,Card,shopping,Low,Starbucks,0
4,TXN0005,U018,2024-01-13,29978.48,Apple,UPI,refund,High,Apple,1
...,...,...,...,...,...,...,...,...,...,...
995,TXN0996,U015,2024-02-22,10566.02,Big Bazaar,Card,refund,Medium-Low,Big Bazaar,1
996,TXN0997,U008,2024-03-13,7355.87,Local Shop,UPI,electronics,Low,Local Shop,0
997,TXN0998,U011,2024-01-16,12636.47,DMart,UPI,refund,Medium-Low,DMart,1
998,TXN0999,U006,2024-01-16,22967.63,Starbucks,UPI,Food & Beverage,Medium-High,Starbucks,0
